# TabSyn Training — NFL Team-Season Wide Table

**What this notebook does:**
1. Installs TabSyn and dependencies
2. Loads `wide_team_seasons.csv` (352 rows × 107 columns)
3. Trains a TabSyn model (VAE encoder + score-based diffusion)
4. Validates that generated samples respect football constraints (e.g. WR TD sum ≤ team passing TDs)
5. Saves model weights to Google Drive for use in the inference microservice

**Before running:** Upload `wide_team_seasons.csv` to your Google Drive and update `CSV_PATH` below.

## 0 — Runtime check
Make sure you're on a GPU runtime: Runtime → Change runtime type → T4 GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Switch to T4 GPU runtime before proceeding.')

## 1 — Install dependencies

In [ ]:
%%capture
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install scikit-learn pandas numpy scipy einops tqdm

# Clone TabSyn from the official repo
import os
if not os.path.exists('TabSyn'):
    !git clone https://github.com/amazon-science/tabsyn.git TabSyn
%cd TabSyn
!pip install -e . --quiet

## 2 — Mount Drive and load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

# ── UPDATE THIS PATH to wherever you uploaded the CSV in your Drive ──────────
CSV_PATH = '/content/drive/MyDrive/nfl_data/wide_team_seasons.csv'
# ────────────────────────────────────────────────────────────────────────────

df_raw = pd.read_csv(CSV_PATH)
print(f'Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
df_raw.head(2)

## 3 — Preprocessing

TabSyn requires:
- All-numeric training matrix (no string columns)
- A metadata dict telling it which columns are continuous vs categorical
- No NaNs

We drop `team` (string) and keep `season` as a continuous conditioning feature.

In [ ]:
# Columns to drop entirely from training
# 'team' is a string label — not useful as a numeric feature
# rs1_* are all zeros (RS position not in nflreadpy player_stats)
DROP_COLS = ['team', 'rs1_kr_att', 'rs1_kr_yds', 'rs1_pr_att', 'rs1_pr_yds']

df = df_raw.drop(columns=[c for c in DROP_COLS if c in df_raw.columns]).copy()
print(f'Training columns: {df.shape[1]}')

# Confirm no nulls
assert df.isnull().sum().sum() == 0, 'NaNs found — fix before training'
print('No NaNs. Good.')

In [ ]:
# Define column types for TabSyn metadata
# Categorical: ls_roster_churn (integer 1-5), season (integer year)
# Everything else: continuous

CATEGORICAL_COLS = ['season', 'ls_roster_churn']
CONTINUOUS_COLS  = [c for c in df.columns if c not in CATEGORICAL_COLS]

print(f'Continuous : {len(CONTINUOUS_COLS)} columns')
print(f'Categorical: {len(CATEGORICAL_COLS)} columns')

# Encode categoricals as integer codes
cat_encoders = {}
for col in CATEGORICAL_COLS:
    unique_vals = sorted(df[col].unique())
    encoder = {v: i for i, v in enumerate(unique_vals)}
    cat_encoders[col] = encoder
    df[col] = df[col].map(encoder)
    print(f'  {col}: {len(unique_vals)} unique values → encoded 0-{len(unique_vals)-1}')

In [ ]:
import json
import os

# Save encoder map to Drive so the inference service can decode samples
SAVE_DIR = '/content/drive/MyDrive/nfl_data/tabsyn_weights'
os.makedirs(SAVE_DIR, exist_ok=True)

# Also save reverse encoders (code → value) for decoding generated samples
# Cast to native Python int — numpy int64 is not JSON-serialisable
reverse_encoders = {
    col: {str(int(code)): int(val) for val, code in enc.items()}
    for col, enc in cat_encoders.items()
}
with open(f'{SAVE_DIR}/cat_encoders.json', 'w') as f:
    json.dump(reverse_encoders, f, indent=2)

# Save column order — inference service needs this to reconstruct rows
col_meta = {
    'all_columns':        list(df.columns),
    'continuous_columns': CONTINUOUS_COLS,
    'categorical_columns': CATEGORICAL_COLS,
    'dropped_columns':    DROP_COLS,
}
with open(f'{SAVE_DIR}/column_meta.json', 'w') as f:
    json.dump(col_meta, f, indent=2)

print('Saved encoders and column metadata to Drive.')

## 4 — Build TabSyn data module

TabSyn's training pipeline expects data in a specific directory structure.
We write it out in the format it expects, then call the standard train scripts.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
import pickle

# Split: 90% train, 10% validation (352 rows — small dataset, keep val small)
df_train, df_val = train_test_split(df, test_size=0.10, random_state=42)
print(f'Train: {len(df_train)}  Val: {len(df_val)}')

# Fit QuantileTransformer on continuous cols (makes distributions more Gaussian
# — TabSyn's diffusion process works better on normalised data)
qt = QuantileTransformer(output_distribution='normal', random_state=42)
X_train_cont = qt.fit_transform(df_train[CONTINUOUS_COLS].values.astype(float))
X_val_cont   = qt.transform(df_val[CONTINUOUS_COLS].values.astype(float))

# Save the scaler — inference service needs it to invert-transform generated samples
with open(f'{SAVE_DIR}/quantile_transformer.pkl', 'wb') as f:
    pickle.dump(qt, f)
print('Saved QuantileTransformer.')

# Categorical arrays
X_train_cat = df_train[CATEGORICAL_COLS].values.astype(int)
X_val_cat   = df_val[CATEGORICAL_COLS].values.astype(int)

# Number of categories per categorical column
cat_dims = [int(df[col].max()) + 1 for col in CATEGORICAL_COLS]
print(f'Cat dims: {dict(zip(CATEGORICAL_COLS, cat_dims))}')

## 5 — Define TabSyn model components

TabSyn = VAE (encoder/decoder) + Score-based diffusion in latent space.
We define both components inline so the notebook is self-contained
(no dependency on TabSyn's internal import structure which can drift).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

N_CONT   = len(CONTINUOUS_COLS)
N_CAT    = len(CATEGORICAL_COLS)
LATENT_D = 256   # VAE latent dimension
HIDDEN_D = 512   # MLP hidden dimension


# ── VAE Encoder ──────────────────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, in_dim, latent_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
        )
        self.mu_head  = nn.Linear(hidden_dim, latent_dim)
        self.log_head = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h  = self.net(x)
        return self.mu_head(h), self.log_head(h)


# ── VAE Decoder ──────────────────────────────────────────────────────────────
class Decoder(nn.Module):
    def __init__(self, latent_dim, out_cont, cat_dims, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
        )
        self.cont_head = nn.Linear(hidden_dim, out_cont)
        # One classification head per categorical column
        self.cat_heads = nn.ModuleList([nn.Linear(hidden_dim, d) for d in cat_dims])

    def forward(self, z):
        h      = self.net(z)
        x_cont = self.cont_head(h)
        x_cat  = [head(h) for head in self.cat_heads]
        return x_cont, x_cat


# ── Score network for diffusion in latent space ───────────────────────────────
class ScoreNet(nn.Module):
    """Predicts the score (denoised latent) at each noise level t."""
    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        # Time embedding (sinusoidal)
        self.time_emb = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.net = nn.Sequential(
            nn.Linear(latent_dim + hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, z_noisy, t):
        t_emb = self.time_emb(t.unsqueeze(-1).float())
        h     = torch.cat([z_noisy, t_emb], dim=-1)
        return self.net(h)


print('Model classes defined.')

## 6 — Train the VAE

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Build cat embedding input: embed each cat column then concatenate with cont
CAT_EMB_DIM = 8  # embedding dim per categorical
cat_embeddings = nn.ModuleList([
    nn.Embedding(d, CAT_EMB_DIM) for d in cat_dims
]).to(DEVICE)

IN_DIM = N_CONT + N_CAT * CAT_EMB_DIM

encoder = Encoder(IN_DIM, LATENT_D, HIDDEN_D).to(DEVICE)
decoder = Decoder(LATENT_D, N_CONT, cat_dims, HIDDEN_D).to(DEVICE)

vae_params = (list(encoder.parameters()) +
              list(decoder.parameters()) +
              list(cat_embeddings.parameters()))
vae_opt = torch.optim.AdamW(vae_params, lr=1e-3, weight_decay=1e-4)
vae_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(vae_opt, T_max=2000)


def build_input(x_cont_t, x_cat_t):
    embs = [cat_embeddings[i](x_cat_t[:, i]) for i in range(N_CAT)]
    return torch.cat([x_cont_t] + embs, dim=-1)


def vae_loss(x_cont_t, x_cat_t, beta=0.001):
    x_in    = build_input(x_cont_t, x_cat_t)
    mu, log = encoder(x_in)
    std     = torch.exp(0.5 * log)
    z       = mu + std * torch.randn_like(std)

    recon_cont, recon_cat = decoder(z)

    # Continuous reconstruction loss (MSE in quantile-normalised space)
    loss_cont = F.mse_loss(recon_cont, x_cont_t)

    # Categorical cross-entropy per column
    loss_cat = sum(
        F.cross_entropy(recon_cat[i], x_cat_t[:, i])
        for i in range(N_CAT)
    ) / max(N_CAT, 1)

    # KL divergence
    kl = -0.5 * (1 + log - mu.pow(2) - log.exp()).mean()

    return loss_cont + loss_cat + beta * kl, loss_cont.item(), loss_cat.item(), kl.item()


# ── Training loop ─────────────────────────────────────────────────────────────
X_cont_tr = torch.FloatTensor(X_train_cont).to(DEVICE)
X_cat_tr  = torch.LongTensor(X_train_cat).to(DEVICE)
X_cont_vl = torch.FloatTensor(X_val_cont).to(DEVICE)
X_cat_vl  = torch.LongTensor(X_val_cat).to(DEVICE)

train_ds = TensorDataset(X_cont_tr, X_cat_tr)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)

VAE_EPOCHS = 2000
best_vae_loss = float('inf')

for epoch in range(1, VAE_EPOCHS + 1):
    encoder.train(); decoder.train(); cat_embeddings.train()
    epoch_loss = 0.0
    for xc, xk in train_dl:
        vae_opt.zero_grad()
        loss, _, _, _ = vae_loss(xc, xk)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vae_params, 1.0)
        vae_opt.step()
        epoch_loss += loss.item()
    vae_scheduler.step()

    if epoch % 200 == 0 or epoch == 1:
        encoder.eval(); decoder.eval(); cat_embeddings.eval()
        with torch.no_grad():
            val_loss, lc, lk, kl = vae_loss(X_cont_vl, X_cat_vl)
        print(f'Epoch {epoch:4d} | train {epoch_loss/len(train_dl):.4f} '
              f'| val {val_loss:.4f} (cont={lc:.4f} cat={lk:.4f} kl={kl:.4f})')
        if val_loss < best_vae_loss:
            best_vae_loss = val_loss
            torch.save({
                'encoder': encoder.state_dict(),
                'decoder': decoder.state_dict(),
                'cat_embeddings': cat_embeddings.state_dict(),
            }, f'{SAVE_DIR}/vae_best.pt')

print(f'VAE training complete. Best val loss: {best_vae_loss:.4f}')

## 7 — Extract latent codes for diffusion training

In [ ]:
# Load best VAE weights
ckpt = torch.load(f'{SAVE_DIR}/vae_best.pt', map_location=DEVICE)
encoder.load_state_dict(ckpt['encoder'])
cat_embeddings.load_state_dict(ckpt['cat_embeddings'])

encoder.eval(); cat_embeddings.eval()
with torch.no_grad():
    x_in_full = build_input(
        torch.FloatTensor(X_train_cont).to(DEVICE),
        torch.LongTensor(X_train_cat).to(DEVICE)
    )
    mu_train, _ = encoder(x_in_full)

Z_train = mu_train.cpu()  # shape: (n_train, LATENT_D)
print('Latent codes shape:', Z_train.shape)

# Normalise latents — diffusion trains better on unit-variance data
z_mean = Z_train.mean(0)
z_std  = Z_train.std(0).clamp(min=1e-6)
Z_norm = (Z_train - z_mean) / z_std

# Save normalisation stats for inference
torch.save({'z_mean': z_mean, 'z_std': z_std}, f'{SAVE_DIR}/latent_norm.pt')
print('Saved latent normalisation stats.')

## 8 — Train the Score/Diffusion model

In [ ]:
score_net = ScoreNet(LATENT_D, HIDDEN_D).to(DEVICE)
diff_opt  = torch.optim.AdamW(score_net.parameters(), lr=1e-3, weight_decay=1e-4)
diff_sched = torch.optim.lr_scheduler.CosineAnnealingLR(diff_opt, T_max=5000)

Z_ds = TensorDataset(Z_norm.to(DEVICE))
Z_dl = DataLoader(Z_ds, batch_size=64, shuffle=True)

# Linear noise schedule (beta_min → beta_max over T steps)
T_STEPS = 1000
beta_min, beta_max = 1e-4, 0.02
betas   = torch.linspace(beta_min, beta_max, T_STEPS).to(DEVICE)
alphas  = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)  # ᾱ_t


def diffusion_loss(z0_batch):
    B = z0_batch.shape[0]
    t = torch.randint(0, T_STEPS, (B,), device=DEVICE)
    noise = torch.randn_like(z0_batch)
    ab    = alpha_bar[t].unsqueeze(1)
    z_t   = torch.sqrt(ab) * z0_batch + torch.sqrt(1 - ab) * noise
    t_frac = t.float() / T_STEPS
    pred_noise = score_net(z_t, t_frac)
    return F.mse_loss(pred_noise, noise)


DIFF_EPOCHS = 5000
best_diff_loss = float('inf')

for epoch in range(1, DIFF_EPOCHS + 1):
    score_net.train()
    epoch_loss = 0.0
    for (z,) in Z_dl:
        diff_opt.zero_grad()
        loss = diffusion_loss(z)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(score_net.parameters(), 1.0)
        diff_opt.step()
        epoch_loss += loss.item()
    diff_sched.step()

    if epoch % 500 == 0 or epoch == 1:
        avg = epoch_loss / len(Z_dl)
        print(f'Diff epoch {epoch:5d} | loss {avg:.6f}')
        if avg < best_diff_loss:
            best_diff_loss = avg
            torch.save(score_net.state_dict(), f'{SAVE_DIR}/score_net_best.pt')

print(f'Diffusion training complete. Best loss: {best_diff_loss:.6f}')

## 9 — DDPM Sampling (reverse diffusion)

Generate N team-season rows by running the reverse diffusion process,
then decoding through the VAE decoder.

In [ ]:
# Load best weights
score_net.load_state_dict(torch.load(f'{SAVE_DIR}/score_net_best.pt', map_location=DEVICE))
ckpt = torch.load(f'{SAVE_DIR}/vae_best.pt', map_location=DEVICE)
decoder.load_state_dict(ckpt['decoder'])
norm  = torch.load(f'{SAVE_DIR}/latent_norm.pt', map_location=DEVICE)
z_mean, z_std = norm['z_mean'].to(DEVICE), norm['z_std'].to(DEVICE)

# Per-column stats from the original CSV (season totals) — used to rescale
# generated output back to the correct distribution after qt.inverse_transform.
# The VAE decoder compresses variance on small datasets; this corrects for that.
_ref = df_raw[[c for c in CONTINUOUS_COLS if c in df_raw.columns]]
TRAIN_CONT_MEAN = _ref.mean().values.astype(float)
TRAIN_CONT_STD  = _ref.std().values.astype(float).clip(min=1e-6)


@torch.no_grad()
def ddpm_sample(n_samples: int, ddpm_steps: int = 200) -> torch.Tensor:
    """Run reverse diffusion, return latents in original (un-normalised) space."""
    score_net.eval(); decoder.eval()
    z = torch.randn(n_samples, LATENT_D, device=DEVICE)

    step_indices = torch.linspace(T_STEPS - 1, 0, ddpm_steps, dtype=torch.long)

    for i, t_idx in enumerate(step_indices):
        t_val   = t_idx.float() / T_STEPS
        t_batch = t_val.expand(n_samples).to(DEVICE)

        pred_noise  = score_net(z, t_batch)
        alpha_t     = alphas[t_idx]
        alpha_bar_t = alpha_bar[t_idx]
        beta_t      = betas[t_idx]

        coef1 = 1.0 / torch.sqrt(alpha_t)
        coef2 = beta_t / torch.sqrt(1 - alpha_bar_t)
        mean  = coef1 * (z - coef2 * pred_noise)

        if t_idx > 0:
            noise = torch.randn_like(z)
            z = mean + torch.sqrt(beta_t) * noise
        else:
            z = mean

    return z * z_std + z_mean


@torch.no_grad()
def latents_to_df(z_samples: torch.Tensor) -> pd.DataFrame:
    """Decode latents → continuous + categorical → DataFrame."""
    decoder.eval()
    recon_cont, recon_cat = decoder(z_samples)

    cont_np = recon_cont.cpu().numpy()
    cont_np = qt.inverse_transform(cont_np)

    # Rescale to match training distribution.
    # qt.inverse_transform maps back to the QuantileTransformer's fitted range,
    # but the VAE decoder still compresses variance — correct by z-scoring the
    # generated batch then re-applying the original mean/std from the CSV.
    gen_mean = cont_np.mean(axis=0)
    gen_std  = cont_np.std(axis=0).clip(min=1e-6)
    cont_np  = (cont_np - gen_mean) / gen_std * TRAIN_CONT_STD + TRAIN_CONT_MEAN

    cont_np = np.maximum(cont_np, 0)

    df_out = pd.DataFrame(cont_np, columns=CONTINUOUS_COLS)

    for i, col in enumerate(CATEGORICAL_COLS):
        codes   = recon_cat[i].argmax(dim=-1).cpu().numpy()
        rev_enc = reverse_encoders[col]
        df_out[col] = [rev_enc.get(str(c), c) for c in codes]

    for col in col_meta['all_columns']:
        if col not in df_out.columns:
            df_out[col] = 0
    df_out = df_out[[c for c in col_meta['all_columns'] if c != 'team']]

    return df_out


print('Sampling functions defined.')

## 10 — Validation: do the generated samples respect football constraints?

In [ ]:
N_VALIDATE = 500
z_samples  = ddpm_sample(N_VALIDATE)
df_gen     = latents_to_df(z_samples)

print(f'Generated {len(df_gen)} team-season rows')
print()

# ── Constraint 1: WR TDs ≤ team_passing_tds + 5 (small slack for data noise) ─
wr_td_sum = df_gen['wr1_receiving_tds'] + df_gen['wr2_receiving_tds'] + df_gen['wr3_receiving_tds']
skill_td_sum = wr_td_sum + df_gen['rb1_receiving_tds'] + df_gen['rb2_receiving_tds'] + \
               df_gen['te1_receiving_tds'] + df_gen['te2_receiving_tds']
violations_td = (skill_td_sum > df_gen['team_passing_tds'] + 5).sum()
print(f'TD constraint violations (skill rec TDs > team_pass_tds + 5): {violations_td}/{N_VALIDATE}')
print(f'  Median WR TD sum     : {wr_td_sum.median():.2f}  (real data: 2-4 per season)')
print(f'  Median team_pass_tds : {df_gen["team_passing_tds"].median():.2f}  (real: ~30)')

# ── Constraint 2: FG made ≤ FG attempted ─────────────────────────────────────
fg_violations = (df_gen['k_fg_made'] > df_gen['k_fg_att'] + 0.5).sum()
print(f'\nFG constraint violations (fg_made > fg_att): {fg_violations}/{N_VALIDATE}')
print(f'  Median k_fg_made : {df_gen["k_fg_made"].median():.1f}  (real: ~26)')
print(f'  Median k_fg_att  : {df_gen["k_fg_att"].median():.1f}   (real: ~32)')

# ── Constraint 3: RB1 carries > RB2 carries (depth ordering) ─────────────────
depth_violations = (df_gen['rb2_carries'] > df_gen['rb1_carries']).sum()
print(f'\nDepth ordering violations (rb2 carries > rb1): {depth_violations}/{N_VALIDATE}')

# ── Constraint 4: WR1 targets > WR2 targets ──────────────────────────────────
wr_order_violations = (df_gen['wr2_targets'] > df_gen['wr1_targets']).sum()
print(f'WR target ordering violations (wr2 > wr1): {wr_order_violations}/{N_VALIDATE}')

# ── Distributions: quick sanity check ────────────────────────────────────────
print('\nGenerated stat ranges (should match real NFL seasons):')
checks = {
    'qb_passing_yards': (3500, 4800),
    'rb1_rushing_yards': (900, 1400),
    'wr1_receiving_yards': (900, 1400),
    'ol_sacks_allowed': (20, 55),
    'p_punt_attempts': (55, 85),
    'k_fg_made': (20, 35),
}
for col, (lo, hi) in checks.items():
    med = df_gen[col].median()
    flag = '✓' if lo <= med <= hi else '⚠️ out of range'
    print(f'  {col:<28} median={med:7.1f}  expected [{lo}-{hi}]  {flag}')

## 11 — Save model config for inference microservice

In [ ]:
model_config = {
    'latent_dim':   LATENT_D,
    'hidden_dim':   HIDDEN_D,
    'cat_emb_dim':  CAT_EMB_DIM,
    'n_continuous': N_CONT,
    'n_categorical': N_CAT,
    'cat_dims':     cat_dims,
    'T_steps':      T_STEPS,
    'beta_min':     beta_min,
    'beta_max':     beta_max,
    'ddpm_sample_steps': 200,
}
with open(f'{SAVE_DIR}/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

print('All artifacts saved to:', SAVE_DIR)
print()
print('Files to download and put in backend/python_backend/tabsyn_weights/:')
import os
for fname in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f'{SAVE_DIR}/{fname}')
    print(f'  {fname:<35} {size/1024:.1f} KB')

## 12 — Download weights

Run this cell to zip all weights and download them. Then put the contents of the zip at:
`backend/python_backend/tabsyn_weights/`

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/tabsyn_weights'
shutil.make_archive(zip_path, 'zip', SAVE_DIR)
files.download(f'{zip_path}.zip')
print('Download started.')